In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Tesla EA Deliveries and Production Data (2015–2025)

## End-to-End Machine Learning Pipeline

### Project Objectives

This project demonstrates a complete Machine Learning pipeline using Tesla deliveries and production data.

The workflow includes:

- Data Loading
- Data Cleaning
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Regression Modeling
- Hyperparameter Tuning
- Model Evaluation
- Time Series Visualization

# Import Required Libraries

The following libraries are imported for:

- Data manipulation
- Data visualization
- Machine Learning
- Model evaluation

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings("ignore")

# Load Dataset

The Tesla dataset is loaded into a Pandas DataFrame for analysis.

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv")

df.head()

# Dataset Overview

In this section, we examine the dataset structure by checking:

- Dataset shape
- Column names
- Data types
- Summary statistics

This helps us understand the dataset before preprocessing.

In [ ]:
print("="*60)
print("DATASET SHAPE")
print("="*60)
print(df.shape)

print("\n")

print("="*60)
print("COLUMN NAMES")
print("="*60)
print(df.columns.tolist())

print("\n")

print("="*60)
print("DATA INFORMATION")
print("="*60)
df.info()

print("\n")

print("="*60)
print("STATISTICAL SUMMARY")
print("="*60)
display(df.describe())

# Data Cleaning

Data cleaning is an essential preprocessing step.

We will check:

- Missing values
- Duplicate records

A clean dataset improves model performance and reliability.

In [ ]:
print("="*60)
print("MISSING VALUES")
print("="*60)

display(df.isnull().sum())

print("\n")

print("="*60)
print("DUPLICATE ROWS")
print("="*60)

print(df.duplicated().sum())

In [ ]:
df.drop_duplicates(inplace=True)

print("Dataset Shape After Removing Duplicates:")
print(df.shape)

# Exploratory Data Analysis (EDA)

EDA helps us understand the distribution, trends, and relationships within the dataset.

The following visualizations analyze:

- Estimated Deliveries by Year
- Production Units by Year
- Tesla Model Distribution
- Correlation between numerical features
- Relationship between Production and Deliveries

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    data=df,
    x="Year",
    y="Estimated_Deliveries",
    estimator=np.mean
)

plt.title("Average Estimated Deliveries by Year")
plt.xlabel("Year")
plt.ylabel("Estimated Deliveries")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.lineplot(
    data=df,
    x="Year",
    y="Production_Units",
    marker="o"
)

plt.title("Production Units Trend")
plt.xlabel("Year")
plt.ylabel("Production Units")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="Model"
)

plt.xticks(rotation=45)

plt.title("Tesla Model Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

correlation = df.select_dtypes(include=np.number).corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=df,
    x="Production_Units",
    y="Estimated_Deliveries",
    hue="Model"
)

plt.title("Production Units vs Estimated Deliveries")

plt.show()

## EDA Summary

From the above visualizations we observe:

- Estimated deliveries vary across different years.
- Production units generally follow a similar trend to deliveries.
- Tesla models are evenly represented in the dataset.
- Numerical variables exhibit varying degrees of correlation.
- Production Units and Estimated Deliveries show a positive relationship.

# Feature Engineering

Feature Engineering improves the dataset by converting categorical variables into numerical values and creating additional features that can help improve machine learning model performance.

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

categorical_columns = [
    "Month",
    "Region",
    "Model",
    "Source_Type"
]

for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

print("Categorical variables encoded successfully.")

df.head()

In [ ]:
# Create previous month's deliveries feature
df["Previous_Deliveries"] = df["Estimated_Deliveries"].shift(1)

# Create rolling mean feature
df["Rolling_Mean_Deliveries"] = (
    df["Estimated_Deliveries"]
    .rolling(window=3)
    .mean()
)

# Remove rows containing NaN values created by shift/rolling
df.dropna(inplace=True)

print("Feature engineering completed.")

df.head()

# Preparing Data for Machine Learning

The dataset is now split into:

- Features (X)
- Target Variable (y)

The target variable is **Estimated_Deliveries**.

In [ ]:
X = df.drop("Estimated_Deliveries", axis=1)

y = df["Estimated_Deliveries"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

# Linear Regression Model

Linear Regression is used as the baseline regression model to predict the Estimated Deliveries based on the available features.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("Linear Regression Model Trained Successfully!")

# Model Evaluation

The model is evaluated using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Linear Regression Performance\n")

print("MAE :", round(mean_absolute_error(y_test, y_pred),2))

print("RMSE :", round(np.sqrt(mean_squared_error(y_test, y_pred)),2))

print("R2 Score :", round(r2_score(y_test, y_pred),4))

# Random Forest Regression

Random Forest is an ensemble learning algorithm that often provides better prediction accuracy than Linear Regression.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Model Trained Successfully!")

# Random Forest Evaluation

In [ ]:
print("Random Forest Performance\n")

print("MAE :", round(mean_absolute_error(y_test, rf_pred),2))

print("RMSE :", round(np.sqrt(mean_squared_error(y_test, rf_pred)),2))

print("R2 Score :", round(r2_score(y_test, rf_pred),4))

# Hyperparameter Tuning

GridSearchCV is used to find the best parameters for the Random Forest model.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

# Feature Importance

Feature importance helps identify which features contribute most to predicting Tesla estimated deliveries.

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

plt.figure(figsize=(10,6))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance")

plt.show()

importance

# Actual vs Predicted Deliveries

The following comparison shows how closely the Linear Regression model predicts the actual delivery values.

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(10)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    comparison["Actual"].values[:50],
    label="Actual"
)

plt.plot(
    comparison["Predicted"].values[:50],
    label="Predicted"
)

plt.title("Actual vs Predicted Deliveries")

plt.legend()

plt.show()

# Time Series Visualization

This visualization illustrates the trend of estimated Tesla deliveries over time.

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(df["Estimated_Deliveries"])

plt.title("Estimated Deliveries Over Time")

plt.xlabel("Record Number")

plt.ylabel("Estimated Deliveries")

plt.grid(True)

plt.show()

# Conclusion

This notebook successfully demonstrates an end-to-end Machine Learning pipeline using the Tesla EA Deliveries and Production dataset.

## Tasks Completed

- Data Loading
- Data Cleaning
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Train-Test Split
- Linear Regression
- Random Forest Regression
- Hyperparameter Tuning using GridSearchCV
- Model Evaluation
- Feature Importance Analysis
- Time Series Visualization

The project demonstrates a complete workflow from raw data preprocessing to predictive modeling and evaluation.